# 02 - Preprocessing et Encodage

## Women's E-Commerce Clothing Reviews

Dans ce notebook, je prépare les données nettoyées pour les algorithmes de classification
de Machine Learning classique.

Le premier notebook (`01_eda_and_cleaning.ipynb`) avait pour objectif de comprendre
et nettoyer les données. Ici, je transforme ces données afin qu'elles puissent être
utilisées par les modèles.

Les principales étapes sont :

- charger le dataset nettoyé ;
- séparer les variables explicatives `X` et la cible `y` ;
- supprimer les identifiants non informatifs ;
- séparer les données en train et test ;
- distinguer les variables numériques, catégorielles et textuelles ;
- préparer les variables numériques ;
- encoder les variables catégorielles ;
- transformer les textes avec TF-IDF ;
- construire un préprocesseur complet avec `ColumnTransformer` ;
- apprendre le prétraitement uniquement sur le jeu d'entraînement ;
- transformer les jeux train et test ;
- vérifier le résultat final.

L'objectif est d'obtenir des données numériques prêtes pour le notebook
`03_classification.ipynb`.

Aucun Deep Learning n'est utilisé dans cette étape.


## 1. Importation des bibliothèques

J'importe les bibliothèques nécessaires pour :

- manipuler les données avec pandas ;
- gérer les chemins avec pathlib ;
- séparer les données en train et test ;
- construire des pipelines de prétraitement ;
- imputer les valeurs manquantes ;
- standardiser les variables numériques ;
- encoder les variables catégorielles ;
- transformer les textes avec TF-IDF.


In [1]:
import warnings

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 2. Chargement du dataset nettoyé

Le premier notebook a produit le fichier :

`data/processed/reviews_clean.csv`

Je charge ce fichier plutôt que le dataset brut, car le nettoyage a déjà été effectué
dans le notebook précédent.


In [ ]:
PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "reviews_clean.csv"
)

df = pd.read_csv(DATA_PATH, keep_default_na=False)

print("Dataset nettoyé chargé avec succès.")
print(f"Chemin : {DATA_PATH}")
print(f"Dimensions : {df.shape}")

print("Valeurs manquantes :")
display(df.isna().sum())


Dataset nettoyé chargé avec succès.
Chemin : ..\data\processed\reviews_clean.csv
Dimensions : (23465, 10)
Valeurs manquantes :


Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64

## 3. Première vérification

Je regarde quelques lignes et les informations générales afin de vérifier que le
dataset correspond bien au résultat du notebook de nettoyage.


In [19]:
display(df.head())

print("\nInformations générales :")
df.info()


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses



Informations générales :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23465 entries, 0 to 23464
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Clothing ID              23465 non-null  int64 
 1   Age                      23465 non-null  int64 
 2   Title                    23465 non-null  object
 3   Review Text              23465 non-null  object
 4   Rating                   23465 non-null  int64 
 5   Recommended IND          23465 non-null  int64 
 6   Positive Feedback Count  23465 non-null  int64 
 7   Division Name            23465 non-null  object
 8   Department Name          23465 non-null  object
 9   Class Name               23465 non-null  object
dtypes: int64(5), object(5)
memory usage: 1.8+ MB


## 4. Définition de la cible

La cible du problème de classification est `Recommended IND`.

Elle possède deux valeurs :

- `0` : la cliente ne recommande pas le produit ;
- `1` : la cliente recommande le produit.

Je sépare donc :

- `X` : les variables utilisées pour effectuer la prédiction ;
- `y` : la variable à prédire.


In [20]:
TARGET_COL = "Recommended IND"

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nDistribution de la cible :")
display(y.value_counts().to_frame("Count"))

print("\nProportion de chaque classe :")
display(
    (y.value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("Percentage")
)


Dimensions de X : (23465, 9)
Dimensions de y : (23465,)

Distribution de la cible :


,Count
Recommended IND,
1,19293
0,4172



Proportion de chaque classe :


,Percentage
Recommended IND,
1,82.22
0,17.78


## 5. Vérification de la cible

Avant d'aller plus loin, je vérifie que la cible contient uniquement les deux classes
attendues : `0` et `1`.

Cette vérification permet de s'assurer qu'il s'agit bien d'une classification binaire.


In [21]:
print("Valeurs présentes dans la cible :")
print(sorted(y.dropna().unique()))

if set(y.dropna().unique()).issubset({0, 1}):
    print("✓ La cible est bien binaire.")
else:
    print("⚠ Vérifier les valeurs présentes dans la cible.")


Valeurs présentes dans la cible :
[np.int64(0), np.int64(1)]
✓ La cible est bien binaire.


## 6. Suppression de l'identifiant

`Clothing ID` identifie un produit, mais ce nombre ne représente pas une quantité
numérique au sens classique.

Par exemple, le produit 1000 n'est pas « plus grand » que le produit 500.

Je retire donc `Clothing ID` du jeu de variables explicatives pour éviter de traiter
un identifiant comme une variable quantitative.


In [23]:
if "Clothing ID" in X.columns:
    X = X.drop(columns=["Clothing ID"])
    print("✓ Clothing ID supprimée.")
else:
    print("✓ Clothing ID n'est pas présente.")

print("\nVariables restantes :")
print(X.columns.tolist())


✓ Clothing ID n'est pas présente.

Variables restantes :
['Age', 'Title', 'Review Text', 'Rating', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']


## 7. Séparation train / test

Je divise les données en deux parties :

- `80 %` pour l'entraînement ;
- `20 %` pour le test.

Le jeu d'entraînement sert à apprendre le modèle et le prétraitement.

Le jeu de test est conservé pour évaluer le modèle sur des données qu'il n'a pas vues
pendant l'apprentissage.

J'utilise `stratify=y` afin de conserver approximativement la même proportion des
classes `0` et `1` dans les deux jeux.


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensions après séparation :")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")


Dimensions après séparation :
X_train : (18772, 8)
X_test  : (4693, 8)
y_train : (18772,)
y_test  : (4693,)


## 8. Vérification de la distribution de la cible après le split

Je vérifie que la proportion des classes est restée proche de celle du dataset initial.

Cette vérification permet de confirmer que `stratify=y` a bien conservé la structure
de la cible dans les deux sous-ensembles.


In [25]:
distribution_split = pd.DataFrame({
    "Train (%)": y_train.value_counts(normalize=True).sort_index() * 100,
    "Test (%)": y_test.value_counts(normalize=True).sort_index() * 100
}).round(2)

display(distribution_split)


,Train (%),Test (%)
Recommended IND,,
0,17.78,17.77
1,82.22,82.23


## 9. Identification des familles de variables

Les variables n'ont pas toutes le même type.

Je les sépare en trois familles :

### Variables numériques

- `Age`
- `Rating`
- `Positive Feedback Count`

### Variables catégorielles

- `Division Name`
- `Department Name`
- `Class Name`

### Variables textuelles

- `Title`
- `Review Text`

Cette séparation est nécessaire car chaque famille nécessite une transformation
différente.


In [26]:
numeric_features = [
    "Age",
    "Rating",
    "Positive Feedback Count"
]

categorical_features = [
    "Division Name",
    "Department Name",
    "Class Name"
]

text_features = [
    "Title",
    "Review Text"
]

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)
print("Variables textuelles :", text_features)


Variables numériques : ['Age', 'Rating', 'Positive Feedback Count']
Variables catégorielles : ['Division Name', 'Department Name', 'Class Name']
Variables textuelles : ['Title', 'Review Text']


## 10. Vérification des colonnes

Je vérifie que toutes les colonnes nécessaires existent bien dans `X_train`.

Cela évite de construire un pipeline avec un nom de colonne incorrect.


In [27]:
all_expected_features = (
    numeric_features
    + categorical_features
    + text_features
)

missing_features = [
    col for col in all_expected_features
    if col not in X_train.columns
]

if not missing_features:
    print("✓ Toutes les variables nécessaires sont présentes.")
else:
    print("⚠ Variables manquantes :", missing_features)


✓ Toutes les variables nécessaires sont présentes.


## 11. Prétraitement des variables numériques

Les variables numériques sont :

- `Age`
- `Rating`
- `Positive Feedback Count`

Je construis un pipeline avec deux opérations :

1. `SimpleImputer(strategy="median")` :
   remplace une éventuelle valeur numérique manquante par la médiane ;

2. `StandardScaler()` :
   standardise les variables afin de les placer sur des échelles comparables.

Même si le notebook précédent ne laisse plus de valeurs numériques manquantes, garder
l'imputation dans le pipeline rend le traitement plus robuste.


In [28]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("✓ Pipeline numérique créé.")
print(numeric_pipeline)


✓ Pipeline numérique créé.
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


## 12. Prétraitement des variables catégorielles

Les variables catégorielles contiennent des modalités comme :

- `General`
- `Dresses`
- `Tops`
- `Knits`
- etc.

Un modèle classique ne peut pas utiliser directement ces chaînes de caractères.

J'utilise donc :

1. `SimpleImputer(strategy="most_frequent")` pour traiter une éventuelle catégorie
   manquante ;

2. `OneHotEncoder(handle_unknown="ignore")` pour transformer chaque catégorie en
   variables numériques 0/1.

`handle_unknown="ignore"` permet au pipeline de fonctionner même si une catégorie
apparaît dans le test alors qu'elle n'était pas présente dans l'entraînement.


In [30]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

print("✓ Pipeline catégoriel créé.")
print(categorical_pipeline)


✓ Pipeline catégoriel créé.
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))])


## 13. Prétraitement du texte avec TF-IDF

Les variables `Title` et `Review Text` contiennent du texte libre.

Un algorithme classique ne peut pas utiliser directement une phrase complète.
Je transforme donc le texte en variables numériques avec **TF-IDF**.

TF-IDF attribue un poids aux mots selon leur importance dans les documents.

J'utilise deux vectoriseurs :

- un pour `Title` ;
- un pour `Review Text`.

J'utilise des unigrammes et bigrammes (`ngram_range=(1, 2)`) afin de conserver des mots
seuls et certaines associations de deux mots.

Je limite également le nombre de caractéristiques pour éviter de créer une matrice
inutilement énorme.


In [31]:
title_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=3000
)

review_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000
)

print("✓ TF-IDF pour Title créé.")
print("✓ TF-IDF pour Review Text créé.")


✓ TF-IDF pour Title créé.
✓ TF-IDF pour Review Text créé.


## 14. Construction du préprocesseur complet

J'ai maintenant quatre traitements différents :

- variables numériques → imputation + standardisation ;
- variables catégorielles → imputation + One-Hot Encoding ;
- `Title` → TF-IDF ;
- `Review Text` → TF-IDF.

`ColumnTransformer` permet d'appliquer chaque transformation uniquement aux colonnes
qui lui correspondent, puis de combiner automatiquement les résultats.


In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        ),
        (
            "title_tfidf",
            title_tfidf,
            "Title"
        ),
        (
            "review_tfidf",
            review_tfidf,
            "Review Text"
        )
    ],
    remainder="drop"
)

print("✓ Préprocesseur complet créé.")
print(preprocessor)


✓ Préprocesseur complet créé.
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Rating', 'Positive Feedback Count']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Division Name', 'Department Name',
                                  'Class Name']),
                                ('title_tfidf',
                                 TfidfVectorizer(max_features=3000,
                                         

## 15. Apprentissage du prétraitement sur le train uniquement

Cette étape est très importante.

J'utilise :

`fit_transform(X_train)`

sur les données d'entraînement.

Le `fit` signifie que le préprocesseur apprend ses paramètres à partir du train :

- médianes ;
- catégories rencontrées ;
- vocabulaire TF-IDF ;
- paramètres de standardisation.

Le test ne doit pas participer à cet apprentissage, afin d'éviter la **fuite de données
(data leakage)**.


In [33]:
X_train_processed = preprocessor.fit_transform(X_train)

print("✓ Prétraitement appris sur le jeu d'entraînement.")
print("Dimensions de X_train après preprocessing :", X_train_processed.shape)


NaN dans X_train :


Age                        0
Title                      0
Review Text                0
Rating                     0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64


NaN dans X_test :


Age                        0
Title                      0
Review Text                0
Rating                     0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64

✓ Prétraitement appris sur le jeu d'entraînement.
Dimensions de X_train après preprocessing : (18772, 13035)


## 16. Transformation du jeu de test

Pour le test, je n'utilise pas `fit`.

J'utilise uniquement :

`transform(X_test)`

Le test est donc transformé avec les paramètres déjà appris sur le train.


In [34]:
X_test_processed = preprocessor.transform(X_test)

print("✓ Jeu de test transformé.")
print("Dimensions de X_test après preprocessing :", X_test_processed.shape)


✓ Jeu de test transformé.
Dimensions de X_test après preprocessing : (4693, 13035)


## 17. Vérification des dimensions

Avant le prétraitement, nous avions un petit nombre de colonnes.

Après One-Hot Encoding et TF-IDF, le nombre de caractéristiques augmente fortement.

C'est normal :

- les catégories deviennent plusieurs colonnes binaires ;
- les mots du vocabulaire TF-IDF deviennent des caractéristiques numériques.

Les matrices train et test doivent avoir exactement le même nombre de colonnes.


In [35]:
print("===== VÉRIFICATION DES DIMENSIONS =====")
print(f"X_train original       : {X_train.shape}")
print(f"X_test original        : {X_test.shape}")
print(f"X_train prétraité      : {X_train_processed.shape}")
print(f"X_test prétraité       : {X_test_processed.shape}")

if X_train_processed.shape[1] == X_test_processed.shape[1]:
    print("✓ Train et test ont le même nombre de caractéristiques.")
else:
    print("⚠ Problème : les dimensions train/test diffèrent.")


===== VÉRIFICATION DES DIMENSIONS =====
X_train original       : (18772, 8)
X_test original        : (4693, 8)
X_train prétraité      : (18772, 13035)
X_test prétraité       : (4693, 13035)
✓ Train et test ont le même nombre de caractéristiques.


## 18. Vérification du format des matrices

Avec TF-IDF, la représentation obtenue est généralement une matrice creuse
(*sparse matrix*).

Cela est normal : la plupart des caractéristiques sont égales à zéro pour une
observation donnée.

Nous gardons donc cette représentation sous forme sparse au lieu de la convertir
inutilement en matrice dense.


In [36]:
print("Type de X_train_processed :", type(X_train_processed))
print("Type de X_test_processed  :", type(X_test_processed))

if hasattr(X_train_processed, "nnz"):
    print("\nNombre de valeurs non nulles dans X_train :", X_train_processed.nnz)
    print(
        "Taux de remplissage :",
        round(
            X_train_processed.nnz
            / (X_train_processed.shape[0] * X_train_processed.shape[1])
            * 100,
            4
        ),
        "%"
    )


Type de X_train_processed : <class 'scipy.sparse._csr.csr_matrix'>
Type de X_test_processed  : <class 'scipy.sparse._csr.csr_matrix'>

Nombre de valeurs non nulles dans X_train : 716736
Taux de remplissage : 0.2929 %


## 19. Récupération des noms des caractéristiques

Je récupère les noms des variables produites par le `ColumnTransformer`.

Cela permet de comprendre les caractéristiques finales envoyées au modèle.

Les caractéristiques proviennent des variables numériques, des variables catégorielles
encodées et du vocabulaire TF-IDF.


In [37]:
feature_names = preprocessor.get_feature_names_out()

print("Nombre total de caractéristiques :", len(feature_names))
print("\nQuelques caractéristiques :")

for i, feature in enumerate(feature_names[:50], start=1):
    print(f"{i}. {feature}")


Nombre total de caractéristiques : 13035

Quelques caractéristiques :
1. num__Age
2. num__Rating
3. num__Positive Feedback Count
4. cat__Division Name_General
5. cat__Division Name_General Petite
6. cat__Division Name_Initmates
7. cat__Division Name_Unknown
8. cat__Department Name_Bottoms
9. cat__Department Name_Dresses
10. cat__Department Name_Intimate
11. cat__Department Name_Jackets
12. cat__Department Name_Tops
13. cat__Department Name_Trend
14. cat__Department Name_Unknown
15. cat__Class Name_Blouses
16. cat__Class Name_Casual bottoms
17. cat__Class Name_Chemises
18. cat__Class Name_Dresses
19. cat__Class Name_Fine gauge
20. cat__Class Name_Intimates
21. cat__Class Name_Jackets
22. cat__Class Name_Jeans
23. cat__Class Name_Knits
24. cat__Class Name_Layering
25. cat__Class Name_Legwear
26. cat__Class Name_Lounge
27. cat__Class Name_Outerwear
28. cat__Class Name_Pants
29. cat__Class Name_Shorts
30. cat__Class Name_Skirts
31. cat__Class Name_Sleep
32. cat__Class Name_Sweaters
33. cat

## 20. Vérification finale du preprocessing

Je réalise une dernière vérification avant de passer au notebook de classification.

Je vérifie :

- que la cible contient toujours 0 et 1 ;
- que train et test ont le même nombre de caractéristiques ;
- que les matrices ne sont pas vides ;
- que le nombre final de caractéristiques correspond aux noms récupérés.


In [38]:
print("===== VÉRIFICATION FINALE =====")

print("Classes de la cible :", sorted(y.dropna().unique()))
print("Shape X_train_processed :", X_train_processed.shape)
print("Shape X_test_processed  :", X_test_processed.shape)
print("Nombre de feature names :", len(feature_names))

checks = {
    "Cible binaire": set(y.dropna().unique()).issubset({0, 1}),
    "Même nombre de features train/test":
        X_train_processed.shape[1] == X_test_processed.shape[1],
    "Train non vide":
        X_train_processed.shape[0] > 0 and X_train_processed.shape[1] > 0,
    "Test non vide":
        X_test_processed.shape[0] > 0 and X_test_processed.shape[1] > 0,
    "Features cohérentes":
        X_train_processed.shape[1] == len(feature_names)
}

for name, result in checks.items():
    print(f"{'✓' if result else '✗'} {name}")

if all(checks.values()):
    print("\n✓ Le prétraitement est prêt pour la classification.")
else:
    print("\n⚠ Des vérifications doivent être corrigées.")


===== VÉRIFICATION FINALE =====
Classes de la cible : [np.int64(0), np.int64(1)]
Shape X_train_processed : (18772, 13035)
Shape X_test_processed  : (4693, 13035)
Nombre de feature names : 13035
✓ Cible binaire
✓ Même nombre de features train/test
✓ Train non vide
✓ Test non vide
✓ Features cohérentes

✓ Le prétraitement est prêt pour la classification.


# 21. Conclusion

Le prétraitement est maintenant terminé.

Nous avons obtenu :

- `X_train_processed` : données d'entraînement transformées ;
- `X_test_processed` : données de test transformées ;
- `y_train` : cible d'entraînement ;
- `y_test` : cible de test.

Les données sont maintenant sous une forme numérique exploitable par les algorithmes
de classification classiques.
